# **Initalize Notebook:**
Downloads PySpark, sets up global variables, allows Google Cloud importing/exporting

In [ ]:
%pip -q install pyspark==4.0.0 google-cloud-storage==3.4.1
%pip install -q google-cloud-dataproc

In [ ]:
PROJECT_ID = "cs131-medicaid"
REGION = "us-central1"
BUCKET = "cs131-medicaid-bucket/"
DATA_URI = "gs://" + BUCKET + "data/"
OUTPUT_URI = "gs://" + BUCKET + "out/"
MEDICAID_URI = DATA_URI + "medicaid-provider-spending.parquet"
MEDICAID_QUALITY_CHECKS = OUTPUT_URI + "quality_checks.csv"

DATAPROC_PARTITIONS = 16

RUN_COUNT = 3

### **Connect Notebook to Dataproc Serverless**
Run these cells to authenticate and link this Colab session directly to your Serverless Spark runtime.

In [ ]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

from google.cloud import storage
client = storage.Client(project=PROJECT_ID)

## **Local Spark Workflow: Complete End-to-End Pipeline**
This cell writes analysis 1 in local pyspark session (imported from previous for easier access).

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

CONNECTOR = str(Path("gcs-connector-3.1.10.jar").resolve())
urlretrieve("https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-3.1.10.jar", CONNECTOR)

import os
import sys
from pyspark.sql import SparkSession, functions as F

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (SparkSession.builder.appName("CS131-Class23")
    .master("local[2]")
    .config("spark.jars", CONNECTOR)
    .config("spark.hadoop.fs.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.gs.auth.type", "APPLICATION_DEFAULT")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version, "Master:", spark.sparkContext.master)

Spark: 4.0.0 Master: local[2]


In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

def analysis_1_local_spark():
  schema = StructType([
      StructField("BILLING_PROVIDER_NPI_NUM", StringType(), True),
      StructField("SERVICING_PROVIDER_NPI_NUM", StringType(), True),
      StructField("HCPCS_CODE", StringType(), True),
      StructField("CLAIM_FROM_MONTH", StringType(), True),
      StructField("TOTAL_PATIENTS", LongType(), True),
      StructField("TOTAL_CLAIM_LINES", LongType(), True),
      StructField("TOTAL_PAID", DoubleType(), True)
  ])

  quality_checks = (
      spark.read
      .option("header", "true")
      .schema(schema)
      .csv(MEDICAID_QUALITY_CHECKS)
  )

  # Aggregate payments and claim lines by HCPCS code and month
  hcpcs_sum = (
      quality_checks
      .groupBy("HCPCS_CODE", "CLAIM_FROM_MONTH")
      .agg(
          F.sum("TOTAL_CLAIM_LINES").alias("TOTAL_CLAIM_LINES_SUM"),
          F.sum("TOTAL_PAID").alias("TOTAL_PAID_SUM")
      )
  )

  paid = F.col("TOTAL_PAID_SUM")
  lines = F.col("TOTAL_CLAIM_LINES_SUM")

  # Preserve DuckDB's floating-point behavior if a denominator is zero
  paid_per_line = (
      F.when(lines == 0, paid * F.lit(float("inf")))
      .otherwise(paid / lines)
  )

  # Calculate the monthly rate from the totals
  monthly = hcpcs_sum.withColumn(
      "PAID_PER_CLAIM_LINE",
      F.round(paid_per_line, 2)
  )

  # Build a ranking within each HCPCS code
  def month_rank(metric, descending=False):
      metric_order = (
          F.col(metric).desc_nulls_last()
          if descending
          else F.col(metric).asc_nulls_last()
      )

      window = Window.partitionBy("HCPCS_CODE").orderBy(
          metric_order,
          F.col("CLAIM_FROM_MONTH").asc_nulls_last()
      )

      return F.row_number().over(window)

  ranked = (
      monthly
      .withColumn(
          "paid_asc_rank",
          month_rank("PAID_PER_CLAIM_LINE")
      )
      .withColumn(
          "paid_desc_rank",
          month_rank("PAID_PER_CLAIM_LINE", descending=True)
      )
      .withColumn(
          "demand_asc_rank",
          month_rank("TOTAL_CLAIM_LINES_SUM")
      )
      .withColumn(
          "demand_desc_rank",
          month_rank("TOTAL_CLAIM_LINES_SUM", descending=True)
      )
  )

  # Extract the winning month for each ranking
  analysis_1 = (
      ranked
      .groupBy("HCPCS_CODE")
      .agg(
          F.max(
              F.when(F.col("paid_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))
          ).alias("LOWEST_PAID_MONTH"),

          F.max(
              F.when(F.col("paid_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))
          ).alias("HIGHEST_PAID_MONTH"),

          F.max(
              F.when(F.col("demand_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))
          ).alias("LOWEST_DEMAND_MONTH"),

          F.max(
              F.when(F.col("demand_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))
          ).alias("HIGHEST_DEMAND_MONTH")
      )
      .orderBy(F.col("HCPCS_CODE").asc_nulls_last())
  )

  print("Saving output...")
  analysis_1.write.mode("overwrite").option("header", "true").csv(OUTPUT_URI + "analysis_1.csv")
  print("Saved:", OUTPUT_URI + "analysis_1.csv")

In [ ]:
#analysis_1_local_spark()

## **Dataproc Serverless Workflow: Complete End-to-End Pipeline**
This cell writes the combined code (Data Loading, Quality Checks, and Analysis 1) into a single PySpark script.

In [ ]:
job_file_path = "combined_dataproc_job.py"

with open(job_file_path, "w") as f:
    f.write(f"""
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
import sys

# 1. Initialize a native Spark session on Dataproc
spark = SparkSession.builder \
    .appName("Dataproc-Medicaid-Pipeline") \
    .config("spark.sql.shuffle.partitions", "{DATAPROC_PARTITIONS}") \
    .getOrCreate()

print("Spark Session Created on Dataproc Cluster!")

# Define URIs dynamically injected from Colab
OUTPUT_URI = "{OUTPUT_URI}"
QUALITY_CHECKS_URI = OUTPUT_URI + "quality_checks.csv"

# Define strict schema to prevent column mismatch
schema = StructType([
    StructField("BILLING_PROVIDER_NPI_NUM", StringType(), True),
    StructField("SERVICING_PROVIDER_NPI_NUM", StringType(), True),
    StructField("HCPCS_CODE", StringType(), True),
    StructField("CLAIM_FROM_MONTH", StringType(), True),
    StructField("TOTAL_PATIENTS", LongType(), True),
    StructField("TOTAL_CLAIM_LINES", LongType(), True),
    StructField("TOTAL_PAID", DoubleType(), True)
])

# 2. Load pre-computed quality checks from GCS robustly
print(f"Loading pre-existing quality checks from: {{QUALITY_CHECKS_URI}}")

# Read raw first to handle header/no-header discrepancy
raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(QUALITY_CHECKS_URI)
)

# If Spark failed to parse headers and assigned default column names like _c0, fallback to positional renaming
if "_c0" in raw_df.columns:
    print("Detected missing header or default column names. Mapping columns positionally...")
    col_names = ["BILLING_PROVIDER_NPI_NUM", "SERVICING_PROVIDER_NPI_NUM", "HCPCS_CODE", "CLAIM_FROM_MONTH", "TOTAL_PATIENTS", "TOTAL_CLAIM_LINES", "TOTAL_PAID"]
    for idx, col_name in enumerate(col_names):
        if idx < len(raw_df.columns):
            raw_df = raw_df.withColumnRenamed(raw_df.columns[idx], col_name)

# Apply casting manually to match the intended schema
quality_checks = (
    raw_df
    .withColumn("TOTAL_PATIENTS", F.col("TOTAL_PATIENTS").cast(LongType()))
    .withColumn("TOTAL_CLAIM_LINES", F.col("TOTAL_CLAIM_LINES").cast(LongType()))
    .withColumn("TOTAL_PAID", F.col("TOTAL_PAID").cast(DoubleType()))
    .filter(F.col("HCPCS_CODE").isNotNull() & (F.col("HCPCS_CODE") != "HCPCS_CODE")) # Filter out header rows that slipped through
)

# 3. Perform Analysis 1
print("Executing Analysis 1 using loaded quality checks...")
hcpcs_sum = (
    quality_checks
    .groupBy("HCPCS_CODE", "CLAIM_FROM_MONTH")
    .agg(
        F.sum("TOTAL_CLAIM_LINES").alias("TOTAL_CLAIM_LINES_SUM"),
        F.sum("TOTAL_PAID").alias("TOTAL_PAID_SUM")
    )
)

paid = F.col("TOTAL_PAID_SUM")
lines = F.col("TOTAL_CLAIM_LINES_SUM")
paid_per_line = F.when(lines == 0, paid * F.lit(float("inf"))).otherwise(paid / lines)

monthly = hcpcs_sum.withColumn("PAID_PER_CLAIM_LINE", F.round(paid_per_line, 2))

def month_rank(metric, descending=False):
    metric_order = F.col(metric).desc_nulls_last() if descending else F.col(metric).asc_nulls_last()
    window = Window.partitionBy("HCPCS_CODE").orderBy(metric_order, F.col("CLAIM_FROM_MONTH").asc_nulls_last())
    return F.row_number().over(window)

ranked = (
    monthly
    .withColumn("paid_asc_rank", month_rank("PAID_PER_CLAIM_LINE"))
    .withColumn("paid_desc_rank", month_rank("PAID_PER_CLAIM_LINE", descending=True))
    .withColumn("demand_asc_rank", month_rank("TOTAL_CLAIM_LINES_SUM"))
    .withColumn("demand_desc_rank", month_rank("TOTAL_CLAIM_LINES_SUM", descending=True))
)

analysis_1 = (
    ranked
    .groupBy("HCPCS_CODE")
    .agg(
        F.max(F.when(F.col("paid_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("LOWEST_PAID_MONTH"),
        F.max(F.when(F.col("paid_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("HIGHEST_PAID_MONTH"),
        F.max(F.when(F.col("demand_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("LOWEST_DEMAND_MONTH"),
        F.max(F.when(F.col("demand_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("HIGHEST_DEMAND_MONTH")
    )
    .orderBy(F.col("HCPCS_CODE").asc_nulls_last())
)

# 4. Save the output directly back to GCS as a single CSV partition
output_gcs_path = OUTPUT_URI + "analysis_1_dataproc.csv"
print(f"Saving analysis output directly to GCS path: {{output_gcs_path}}")
analysis_1.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_gcs_path)

print("DATAPROC PIPELINE COMPLETE SUCCESSFULLY!")
""")

print(f"Successfully wrote updated combined script locally: {job_file_path}")

Successfully wrote updated combined script locally: combined_dataproc_job.py


### **Optimized Dataproc Pipeline Script**
This optimized version applies `.repartition(X, "HCPCS_CODE")` and `coalesce(8)` when writing the CSV

In [ ]:
job_file_path = "combined_dataproc_job.py"

with open(job_file_path, "w") as f:
    f.write(f"""
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
import sys

# 1. Initialize an optimized Spark session on Dataproc
spark = SparkSession.builder \
    .appName("Dataproc-Medicaid-Pipeline") \
    .config("spark.sql.shuffle.partitions", "{DATAPROC_PARTITIONS}") \
    .getOrCreate()

print("Optimized Spark Session Created on Dataproc Cluster!")

# Define URIs dynamically injected from Colab
OUTPUT_URI = "{OUTPUT_URI}"
QUALITY_CHECKS_URI = OUTPUT_URI + "quality_checks.csv"

# Define strict schema to prevent column mismatch
schema = StructType([
    StructField("BILLING_PROVIDER_NPI_NUM", StringType(), True),
    StructField("SERVICING_PROVIDER_NPI_NUM", StringType(), True),
    StructField("HCPCS_CODE", StringType(), True),
    StructField("CLAIM_FROM_MONTH", StringType(), True),
    StructField("TOTAL_PATIENTS", LongType(), True),
    StructField("TOTAL_CLAIM_LINES", LongType(), True),
    StructField("TOTAL_PAID", DoubleType(), True)
])

# 2. Load pre-computed quality checks from GCS robustly
print(f"Loading pre-existing quality checks from: {{QUALITY_CHECKS_URI}}")

raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(QUALITY_CHECKS_URI)
)

# If Spark failed to parse headers, fallback to positional renaming
if "_c0" in raw_df.columns:
    print("Detected missing header or default column names. Mapping columns positionally...")
    col_names = ["BILLING_PROVIDER_NPI_NUM", "SERVICING_PROVIDER_NPI_NUM", "HCPCS_CODE", "CLAIM_FROM_MONTH", "TOTAL_PATIENTS", "TOTAL_CLAIM_LINES", "TOTAL_PAID"]
    for idx, col_name in enumerate(col_names):
        if idx < len(raw_df.columns):
            raw_df = raw_df.withColumnRenamed(raw_df.columns[idx], col_name)

# Apply casting manually and filter out invalid rows
quality_checks = (
    raw_df
    .withColumn("TOTAL_PATIENTS", F.col("TOTAL_PATIENTS").cast(LongType()))
    .withColumn("TOTAL_CLAIM_LINES", F.col("TOTAL_CLAIM_LINES").cast(LongType()))
    .withColumn("TOTAL_PAID", F.col("TOTAL_PAID").cast(DoubleType()))
    .filter(F.col("HCPCS_CODE").isNotNull() & (F.col("HCPCS_CODE") != "HCPCS_CODE"))
)

# 3. Perform Analysis 1 with Optimization
print("Executing Optimized Analysis 1 using loaded quality checks...")

# OPTIMIZATION: Repartition on the grouping / partition key (HCPCS_CODE)
# This eliminates subsequent multi-stage shuffles for groupby and window computations
optimized_df = quality_checks.repartition({DATAPROC_PARTITIONS}, "HCPCS_CODE")

hcpcs_sum = (
    optimized_df
    .groupBy("HCPCS_CODE", "CLAIM_FROM_MONTH")
    .agg(
        F.sum("TOTAL_CLAIM_LINES").alias("TOTAL_CLAIM_LINES_SUM"),
        F.sum("TOTAL_PAID").alias("TOTAL_PAID_SUM")
    )
)

paid = F.col("TOTAL_PAID_SUM")
lines = F.col("TOTAL_CLAIM_LINES_SUM")
paid_per_line = F.when(lines == 0, paid * F.lit(float("inf"))).otherwise(paid / lines)

monthly = hcpcs_sum.withColumn("PAID_PER_CLAIM_LINE", F.round(paid_per_line, 2))

def month_rank(metric, descending=False):
    metric_order = F.col(metric).desc_nulls_last() if descending else F.col(metric).asc_nulls_last()
    # Using our pre-partitioned key ensures this window runs smoothly without shuffling again
    window = Window.partitionBy("HCPCS_CODE").orderBy(metric_order, F.col("CLAIM_FROM_MONTH").asc_nulls_last())
    return F.row_number().over(window)

ranked = (
    monthly
    .withColumn("paid_asc_rank", month_rank("PAID_PER_CLAIM_LINE"))
    .withColumn("paid_desc_rank", month_rank("PAID_PER_CLAIM_LINE", descending=True))
    .withColumn("demand_asc_rank", month_rank("TOTAL_CLAIM_LINES_SUM"))
    .withColumn("demand_desc_rank", month_rank("TOTAL_CLAIM_LINES_SUM", descending=True))
)

analysis_1 = (
    ranked
    .groupBy("HCPCS_CODE")
    .agg(
        F.max(F.when(F.col("paid_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("LOWEST_PAID_MONTH"),
        F.max(F.when(F.col("paid_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("HIGHEST_PAID_MONTH"),
        F.max(F.when(F.col("demand_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("LOWEST_DEMAND_MONTH"),
        F.max(F.when(F.col("demand_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("HIGHEST_DEMAND_MONTH")
    )
    .orderBy(F.col("HCPCS_CODE").asc_nulls_last())
)

# 4. Save output
output_gcs_path = OUTPUT_URI + "analysis_1_dataproc.csv"
print(f"Saving analysis output directly to GCS path: {{output_gcs_path}}")
analysis_1.coalesce(8).write.mode("overwrite").option("header", "true").csv(output_gcs_path)

print("DATAPROC PIPELINE COMPLETE SUCCESSFULLY!")
""")

print(f"Successfully wrote updated combined script locally: {job_file_path}")

Successfully wrote updated combined script locally: combined_dataproc_job.py


## **Full Dataproc vs Local Spark Benchmark**

In [ ]:
from google.cloud import dataproc_v1 as dataproc

# Instantiate the correct Dataproc Serverless Batch Client
dataproc_client = dataproc.BatchControllerClient(
    client_options={"api_endpoint": f"{REGION}-dataproc.googleapis.com:443"}
)

from google.cloud import storage
gcs_client = storage.Client(project=PROJECT_ID)
bucket = gcs_client.bucket(BUCKET.rstrip("/"))
blob = bucket.blob("code/combined_dataproc_job.py")
blob.upload_from_filename(job_file_path)

gcs_job_uri = f"gs://{BUCKET.rstrip('/')}/code/combined_dataproc_job.py"
print(f"Uploaded combined pipeline script to: {gcs_job_uri}")

Uploaded combined pipeline script to: gs://cs131-medicaid-bucket/code/combined_dataproc_job.py


In [ ]:
import time
import pandas as pd

benchmark_results = []

for run in range(1, RUN_COUNT + 1):
  start_time = time.time()

  # Programmatically submit the Serverless Batch Job to GCP
  batch_id = f"medicaid-pipeline-job-{int(time.time())}"

  batch_request = {
      "pyspark_batch": {
          "main_python_file_uri": gcs_job_uri,
          "args": []
      },
      "runtime_config": {
          "properties": {
              # Enable Adaptive Query Execution for runtime plans optimization
              "spark.sql.adaptive.enabled": "true",
              "spark.sql.adaptive.coalescePartitions.enabled": "true",
              # Use the high-performance Kryo Serializer for shuffling
              "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
              # Enable dynamic allocation to dynamically scale executors automatically
              "spark.dynamicAllocation.enabled": "true",
              "spark.dynamicAllocation.maxExecutors": "4"
          }
      },
      "environment_config": {
          "execution_config": {
              "subnetwork_uri": "default"
          }
      }
  }

  print(f"Submitting Serverless Batch Job '{batch_id}' to Dataproc...")
  operation = dataproc_client.create_batch(
      parent=f"projects/{PROJECT_ID}/locations/{REGION}",
      batch=batch_request,
      batch_id=batch_id
  )

  print(f"Actively polling Dataproc API for job status of {batch_id}...")
  batch_name = f"projects/{PROJECT_ID}/locations/{REGION}/batches/{batch_id}"

  # Active polling mechanism
  pending = True
  while True:
      try:
          batch_status = dataproc_client.get_batch(name=batch_name)
          state = batch_status.state.name
          print(f"Current State: {state}")

          if state == "RUNNING" and pending:
            start_time_execution = time.time()
            pending = False

          if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
              break
      except Exception as e:
          print(f"Polling encountered an update warning: {e}")

      time.sleep(5)

  elapsed = time.time() - start_time
  elapsed_execution = time.time() - start_time_execution
  print(f"Dataproc Job {run} finished with state {state} in {elapsed:.2f} seconds and {elapsed_execution:.2f} seconds exclusively for execution!\n")

  # Get row count from the generated file if succeeded
  if state == "SUCCEEDED":
      input_rows = spark.read.option("header", "true").csv(MEDICAID_URI).count()
      output_rows = spark.read.option("header", "true").csv(OUTPUT_URI + "analysis_1_dataproc.csv").count()
  else:
      input_rows = 0
      output_rows = 0

  benchmark_results.append({
    "approach": "Dataproc Serverless",
    "run": run,
    "input_rows": input_rows,
    "elapsed_seconds": round(elapsed, 4),
    "elapsed_seconds (without runtime startup)": round(elapsed_execution, 4),
    "output_rows": output_rows,
    "job_id": batch_id
  })

for run in range(1, RUN_COUNT + 1):
  start_time = time.time()

  print(f"Local Spark Job {run} called...")
  analysis_1_local_spark()

  elapsed = time.time() - start_time
  print(f"Local Spark Job {run} finished in {elapsed:.2f} seconds!\n")

  input_rows = spark.read.option("header", "true").csv(MEDICAID_URI).count()
  output_rows = spark.read.option("header", "true").csv(OUTPUT_URI + "analysis_1.csv").count()

  benchmark_results.append({
    "approach": "Local Spark",
    "run": run,
    "input_rows": input_rows,
    "elapsed_seconds": round(elapsed, 4),
    "elapsed_seconds (without runtime startup)": "N/A",
    "output_rows": output_rows,
    "job_id": "N/A"
  })

benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)

Submitting Serverless Batch Job 'medicaid-pipeline-job-1794800677' to Dataproc...
Actively polling Dataproc API for job status of medicaid-pipeline-job-1794800677...
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: PENDING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: RUNNING
Current State: SUCCEEDED
Dataproc Job 1 finished with state SUCCEEDED in 143.70 seconds and 71.18 seconds exclusively for execution!

Submitting Serverless Batch Job 'medicaid-pipeline-job-1

,approach,run,input_rows,elapsed_seconds,elapsed_seconds (without runtime startup),output_rows,job_id
0,Dataproc Serverless,1,23418753,143.7028,71.1759,4913,medicaid-pipeline-job-1794800677
1,Dataproc Serverless,2,23418753,143.1988,76.21,4913,medicaid-pipeline-job-1794800881
2,Dataproc Serverless,3,23418753,148.3456,76.2296,4913,medicaid-pipeline-job-1794801083
3,Local Spark,1,23418753,166.4783,N/A,4913,N/A
4,Local Spark,2,23418753,165.8163,N/A,4913,N/A
5,Local Spark,3,23418753,166.9055,N/A,4913,N/A


In [ ]:
# Load the Dataproc output
dataproc_output_path = OUTPUT_URI + "analysis_1_dataproc.csv"
df_dataproc = spark.read.option("header", "true").csv(dataproc_output_path)

# Load the Local Spark output
local_output_path = OUTPUT_URI + "analysis_1.csv"
df_local = spark.read.option("header", "true").csv(local_output_path)

dataproc_count = df_dataproc.count()
local_count = df_local.count()

print(f"Dataproc output row count: {dataproc_count}")
print(f"Local Spark output row count: {local_count}")

# Compute differences using subtract
diff_dataproc_minus_local = df_dataproc.subtract(df_local)
diff_local_minus_dataproc = df_local.subtract(df_dataproc)

dataproc_only_count = diff_dataproc_minus_local.count()
local_only_count = diff_local_minus_dataproc.count()

print(f"Rows in Dataproc but not in Local: {dataproc_only_count}")
print(f"Rows in Local but not in Dataproc: {local_only_count}")

is_identical = (dataproc_only_count == 0 and local_only_count == 0 and dataproc_count == local_count)

# Save the metadata and results of comparison to a CSV file
import pandas as pd

comparison_results = {
    "Metric": [
        "Dataproc Row Count",
        "Local Spark Row Count",
        "Rows Only in Dataproc",
        "Rows Only in Local",
        "Full Equivalence Match"
    ],
    "Value": [
        str(dataproc_count),
        str(local_count),
        str(dataproc_only_count),
        str(local_only_count),
        str(is_identical)
    ]
}

comparison_df = pd.DataFrame(comparison_results)
comparison_df.to_csv(OUTPUT_URI + "result_comparison.csv", index=False)
print("\nSUCCESS: Created 'result_comparison.csv' with the full-result equivalence check metrics.")

if is_identical:
    print("Both datasets are completely identical!")
else:
    print("Warning: Differences detected between Dataproc and Local datasets!")

Dataproc output row count: 4913
Local Spark output row count: 4913
Rows in Dataproc but not in Local: 0
Rows in Local but not in Dataproc: 0

SUCCESS: Created 'result_comparison.csv' with the full-result equivalence check metrics.
Both datasets are completely identical!
